# 01 — Data ingestion

Pull historical daily bars for the strategy universe and cache to parquet + SQLite.

Default source: yfinance. Override to Polygon/Alpaca by changing the client.

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.data.yfinance_client import YFinanceClient
yf = YFinanceClient()
assert yf.available(), 'yfinance not installed'

ETF_UNIVERSE = ['SPY','QQQ','IWM','TLT','IEF','GLD','SLV','USO','DBA',
                'VNQ','HYG','LQD','UUP','FXE','FXY','XLE','XLK']

data = {}
for s in ETF_UNIVERSE:
    df = yf.get_daily_bars(s, start='2010-01-01')
    if not df.empty:
        data[s] = df['adj_close']
        print(f'  {s}: {len(df)} rows {df.index.min().date()} → {df.index.max().date()}')

prices = pd.DataFrame(data).dropna(how='all').ffill().dropna()
prices.to_parquet('../data/parquet/etf_panel.parquet')
print(f'\nPanel: {prices.shape}, cached to data/parquet/etf_panel.parquet')
prices.tail()

### Quick chart

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
(prices / prices.iloc[0]).plot(ax=ax, linewidth=0.8)
ax.set_title('ETF universe (normalised)')
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=8); plt.show()

### Optional: FRED macro series

If you have a FRED key:

In [ ]:
from src.config import KEYS
if KEYS.fred:
    from src.data.fred_client import FREDClient
    fred = FREDClient()
    cpi = fred.get_series('CPIAUCSL', start='2010-01-01')
    print('CPI rows:', len(cpi))
    cpi.tail()
else:
    print('No FRED key set; skipping macro pull')

### Optional: crypto funding history

Uses Binance's public REST endpoint — no auth required.

In [ ]:
from src.data.crypto_clients import BinanceClient
bnc = BinanceClient()
f = bnc.get_funding_history('BTCUSDT', start='2022-01-01')
print(f'BTC funding rows: {len(f)} (last 5):')
f.tail()